# Scriptor: my notes -> my handwriting model

Photos of handwritten pages -> pen strokes (Google InkSight) -> stroke-3 dataset ->
LSTM-MDN (stage A) -> LoRA adapter (stage B) -> generated lines -> G-code for the simulator.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. Then run the cells in order
(or `Runtime -> Run all`; cell 2 waits for you to pick the photos unless they are in
`MyDrive/scriptor_photos`).

Everything is saved to `MyDrive/Scriptor/notes_run/`. All numbers printed and saved are measured in
this run; nothing is filled in by hand.

In [ ]:
# 1. Setup: clone the repo, install, check the GPU, mount Drive
BRANCH = "notes-to-model"  # use "main" once the pull request is merged

import json, os, shutil, subprocess, sys, threading, time
from pathlib import Path

RUN_T0 = time.time()
!rm -rf /content/scriptor && git clone -q -b {BRANCH} https://github.com/Machahary07/scriptor.git /content/scriptor
!pip -q install "Hershey-Fonts>=2.1"
%cd /content/scriptor
sys.path.insert(0, "/content/scriptor")

import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again."
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| torch", torch.__version__)

from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/Scriptor/notes_run")
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)


def run_live(cmd, timeout_s=None, env=None, log_path=None):
    """Run a command, print its output as it arrives and keep a copy. Returns (exit code or 'timeout', output)."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    timer = threading.Timer(timeout_s, p.kill) if timeout_s else None
    if timer:
        timer.start()
    t0, lines = time.time(), []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    code = p.wait()
    if timer:
        timer.cancel()
    out = "".join(lines)
    if log_path:
        Path(log_path).write_text(out)
    return ("timeout" if timeout_s and time.time() - t0 >= timeout_s else code), out

In [ ]:
# 2. Photos: taken from MyDrive/scriptor_photos if that folder has images, otherwise you pick them now
PHOTOS = Path("/content/photos")
shutil.rmtree(PHOTOS, ignore_errors=True)
PHOTOS.mkdir()
EXT = {".jpg", ".jpeg", ".png", ".webp"}
DRIVE_PHOTOS = Path("/content/drive/MyDrive/scriptor_photos")
found = sorted(p for p in DRIVE_PHOTOS.glob("*") if p.suffix.lower() in EXT) if DRIVE_PHOTOS.exists() else []
if found:
    for p in found:
        shutil.copy(p, PHOTOS / p.name)
    print(f"copied {len(found)} photos from {DRIVE_PHOTOS}")
else:
    from google.colab import files
    print("Pick your page photos (for example scriptor/docs/samples/*.jpeg on your laptop)")
    for name, data in files.upload().items():
        (PHOTOS / name).write_bytes(data)
photos = sorted(p for p in PHOTOS.iterdir() if p.suffix.lower() in EXT)
print(len(photos), "photos:", [p.name for p in photos])
assert photos, "no photos"

## Part 1: photo -> strokes

InkSight is TensorFlow, so it gets its own Python environment (`/content/ink_env`) and runs as a
separate process; it never shares memory with PyTorch. Setup is capped at 20 minutes. If it fails,
the cell stops with the exact error. **No strokes are ever faked.**

In [ ]:
# 3. Build the InkSight + docTR environment and test-load the model (20 minute limit)
INK_ENV = "/content/ink_env"
INK_PINS = "repo"  # "repo" = TF 2.20 (official InkSight notebook). If this cell fails, set "card" (TF 2.17) and rerun.

t = time.time()
code, log = run_live(["bash", "engine/photo2ink/setup_env.sh", INK_ENV], timeout_s=20 * 60,
                     env={"INK_PINS": INK_PINS}, log_path=OUT / "inksight_setup.log")
INK_SETUP_S = round(time.time() - t)
if code != 0:
    raise SystemExit(
        f"InkSight setup FAILED (exit {code}) after {INK_SETUP_S} s. The exact error is above and in "
        f"{OUT}/inksight_setup.log.\nTry: (1) INK_PINS = 'card' and rerun this cell; "
        "(2) Runtime -> Disconnect and delete runtime, then run all again; (3) send the log to whoever helps you.")
print(f"InkSight ready in {INK_SETUP_S} s")

In [ ]:
# 4. Photos -> strokes: clean pages, docTR finds and reads words, InkSight traces each word, words -> lines
P2I = OUT / "photo2ink"
code, _ = run_live([f"{INK_ENV}/bin/python", "-m", "engine.photo2ink.run", "--photos", str(PHOTOS), "--out", str(P2I)],
                   log_path=OUT / "photo2ink.log")
if code != 0:
    raise SystemExit(f"photo -> strokes failed (exit {code}); see the output above and {OUT}/photo2ink.log")

from IPython.display import Image as ShowImage, display
report = json.loads((P2I / "photo2ink_report.json").read_text())
for p in sorted((P2I / "previews").glob("*.png"))[:3]:
    display(ShowImage(str(p), width=1100))
print("all previews:", P2I / "previews", "| every word with its flags:", P2I / "words.json")

## Part 2: dataset

Same preprocessing as the rest of Scriptor (`training/data.py`): slant and x-height normalised,
resampled, RDP-simplified, stroke-3 `(dx, dy, pen_lift)`. 90/10 split by line *before*
augmentation; training lines get 3 jittered copies (scale, slant, rotation).

In [ ]:
# 5. writer_samples.json -> stroke-3 dataset
from training.data import norm_stats
from training.notes_dataset import build, load_lines

DS = OUT / "dataset"
ds_stats = build(P2I / "writer_samples.json", DS, max_points=1000, copies=3)
print(json.dumps(ds_stats, indent=1))
train, train_orig, val = (load_lines(DS / f) for f in ("train.npz", "train_orig.npz", "val.npz"))
norm = norm_stats(train)

## Training

**Stage A** trains the full LSTM-MDN (`engine/models/graves_net.py`, about 4.5M parameters) from
scratch on your augmented lines: Adam 1e-3, gradient clip 10, 15 minutes.
**Stage B** freezes it and trains rank-8 LoRA adapters on the LSTM gates, the attention window and
the output head (`engine/models/lora.py`) on your original, un-jittered lines: Adam 5e-4, 10 minutes.
Validation NLL is measured every minute; the best checkpoint / adapter is written to Drive each time it improves.

Note: there is no public pre-training set in this run, so the base model only sees your notes. With
IAM-OnDB for stage A (see `training/scriptor_colab.ipynb`) the adapter has a real general model to adapt.

In [ ]:
# 6. Stage A: base LSTM-MDN
from engine.models.graves_net import GravesNet, save_checkpoint
from training.notes_train import count_params, train_timeboxed

STAGE_A_MIN = 15
torch.manual_seed(0)
net = GravesNet().to("cuda")
BASE = OUT / "graves_base.pt"


def save_base(v, step):
    save_checkpoint(BASE, net, norm, {"val_loss": v, "step": step, "source": "notes_to_model stage A"})


res_a = train_timeboxed(net, train, val, norm, minutes=STAGE_A_MIN, lr=1e-3, clip=10.0, stage="A", on_best=save_base)
(OUT / "history_stage_A.json").write_text(json.dumps(res_a, indent=1))
PARAMS_TOTAL = count_params(net)[1]
print(f"stage A: {res_a['steps']} steps in {res_a['wall_s'] / 60:.1f} min, best val NLL {res_a['best_val']:.4f}, "
      f"{PARAMS_TOTAL:,} parameters")

In [ ]:
# 7. Stage B: freeze the base, train LoRA adapters
from engine.models.graves_net import load_checkpoint
from engine.models.lora import DEFAULT_TARGETS, apply_lora, save_adapter

STAGE_B_MIN, RANK, ALPHA = 10, 8, 16.0
net, norm = load_checkpoint(BASE, "cuda")  # best stage A weights
apply_lora(net, RANK, ALPHA)
net.to("cuda")
TRAINABLE, TOTAL = count_params(net)
print(f"LoRA on {DEFAULT_TARGETS}: trainable {TRAINABLE:,} / total {TOTAL:,} parameters "
      f"({100 * TRAINABLE / TOTAL:.2f}%)")
ADAPTER = OUT / "adapter.pt"


def save_lora(v, step):
    save_adapter(ADAPTER, net, RANK, ALPHA, {"writer": "me", "val_loss": v, "step": step})


res_b = train_timeboxed(net, train_orig, val, norm, minutes=STAGE_B_MIN, lr=5e-4, clip=10.0, stage="B",
                        on_best=save_lora, t_offset=res_a["wall_s"] / 60)
(OUT / "history_stage_B.json").write_text(json.dumps(res_b, indent=1))
ADAPTER_BYTES = ADAPTER.stat().st_size
print(f"stage B: {res_b['steps']} steps in {res_b['wall_s'] / 60:.1f} min, best val NLL {res_b['best_val']:.4f}, "
      f"adapter file {ADAPTER_BYTES / 1024:.0f} KB")

In [ ]:
# 8. Loss curve (both stages, train vs validation)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
for res, c in ((res_a, "tab:blue"), (res_b, "tab:orange")):
    h = res["history"]
    tr = [x for x in h if x["train"] is not None]
    ax.plot([x["t_min"] for x in tr], [x["train"] for x in tr], "-", color=c, label=f"stage {res['stage']} train")
    ax.plot([x["t_min"] for x in h], [x["val"] for x in h], "--o", ms=3, color=c, label=f"stage {res['stage']} validation")
ax.axvline(res_a["wall_s"] / 60, color="grey", lw=0.8)
ax.set_xlabel("training time (minutes)")
ax.set_ylabel("NLL per point (lower is better)")
ax.set_title(f"Stage A (base) then stage B (LoRA r={RANK}), {GPU}")
ax.grid(alpha=0.3)
ax.legend()
fig.savefig(OUT / "loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 9. Generate lines with base + adapter at bias 0 and 2 -> PNG, and G-code through the engine's writer
from engine.models import Style
from engine.models.graves import GravesModel
from engine.pipeline import make_gcode
from engine.planner import PageSpec, plan_pages, render_page

TEXTS = ["linear search with execution time", "Enter element to search", "Element found at position"]
model = GravesModel(weights=BASE)
style = Style(writer_id="me", samples=[l.s3 for l in train_orig[:8]], adapter_path=str(ADAPTER))
spec = PageSpec()
GENERATED = {}
for bias in (0.0, 2.0):
    t = time.time()
    layout = plan_pages("\n".join(TEXTS), model, style, spec)[0]
    paths = render_page(layout, model, style, bias, seed=0)
    page = make_gcode(paths, spec.height, tag=f"notes_to_model bias={bias:g}")
    name = f"generated_bias{bias:g}"
    (OUT / f"{name}.gcode").write_text(page.gcode)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for p in paths:
        ax.plot(p[:, 0], p[:, 1], "k-", lw=0.8)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(f"generated, bias {bias:g} (higher bias = neater, less varied)")
    fig.savefig(OUT / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()
    GENERATED[name] = {"bias": bias, "texts": TEXTS, "seconds": round(time.time() - t, 1),
                       "png": f"{name}.png", "gcode": f"{name}.gcode", "gcode_stats": page.stats}
    print(name, json.dumps(page.stats))
print("load the .gcode files in the web app's Simulator page")

In [ ]:
# 10. run_summary.json: only values measured in this run
summary = {
    "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "gpu": GPU,
    "torch": torch.__version__,
    "photos": report["photos"],
    "words_detected": report["words_detected"],
    "words_kept": report["words_kept"],
    "word_flags": report["word_flags"],
    "lines_total": report["lines_total"],
    "lines_kept": report["lines_kept"],
    "lines_dropped": report["lines_dropped"],
    "inksight": {"model": report["inksight_model"], "load_method": report["inksight_load_method"],
                 "setup_s": INK_SETUP_S, "versions": report["versions"], "timing_s": report["timing_s"]},
    "dataset": ds_stats,
    "params_total": PARAMS_TOTAL,
    "stage_A": {"minutes_budget": STAGE_A_MIN, "lr": 1e-3, "clip": 10.0,
                **{k: res_a[k] for k in ("steps", "epochs", "wall_s", "initial_val", "best_val", "final_train", "final_val")}},
    "stage_B": {"minutes_budget": STAGE_B_MIN, "lr": 5e-4, "clip": 10.0, "rank": RANK, "alpha": ALPHA,
                "targets": list(DEFAULT_TARGETS), "trainable_params": TRAINABLE, "total_params": TOTAL,
                "adapter_bytes": ADAPTER_BYTES,
                **{k: res_b[k] for k in ("steps", "epochs", "wall_s", "initial_val", "best_val", "final_train", "final_val")}},
    "generated": GENERATED,
    "notebook_wall_s": round(time.time() - RUN_T0),
}
(OUT / "run_summary.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
print("\nfiles in", OUT, ":", sorted(p.name for p in OUT.iterdir()))

## Done

In `MyDrive/Scriptor/notes_run/`: `photo2ink/previews/`, `loss_curve.png`, `generated_bias0.png`,
`generated_bias2.png`, the two `.gcode` files, `run_summary.json`, `graves_base.pt` and `adapter.pt`.

To use the model in the local app: copy `graves_base.pt` to `scriptor/data/models/` and `adapter.pt` to
`scriptor/data/writers/<writer-id>/adapter.pt`.